# 04 Land classification (clustering)

Phase 5 (Tasks P5.1–P5.7, PRD FR-5, §9.4). Groups the 100 m cells into **built-up, forest and usable** land.

1. Inputs and the k scan (K-Means vs GMM): why **K-Means, k = 7**.
2. Cluster profiles and a description of each cluster.
3. Cluster → class rules (`src/classify/label.py`) and the exclusion mask.
4. The 3-class maps for both runs: **validation** (2018 data only, time-travel rule) and **final** (2023).
5. Agreement with ESRI (primary) and WorldCover 2021 (cross-check).
6. A visual check against satellite imagery at 5 random locations.

Pipeline (skips nothing, ~3 min):
`python -m src.classify.cluster` → `python -m src.classify.label` → `python -m src.classify.export` → `python -m src.classify.evaluate`.

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

from src.classify import cluster as cl
from src.classify import label as lb
from src.classify.evaluate import NAMES, eval_path
from src.config import load_config
from src.features import schema

cfg = load_config()
FIG = Path(cfg.paths["figures"])
FIG.mkdir(parents=True, exist_ok=True)
metrics = json.loads(cl.metrics_path(cfg).read_text())
c7 = {run: pd.read_parquet(lb.c7_path(cfg, run)) for run in cl.RUNS}
rules = {run: pd.read_csv(lb.rules_path(cfg, run), index_col="cluster_id") for run in cl.RUNS}
grid = schema.read_contract(schema.contract_path(cfg, "C1"), "C1")
CLASS_COLORS = {1: "#d7301f", 2: "#1a9850", 3: "#fee08b", 255: "#9ecae1"}
print("inputs:", metrics["final"]["inputs"])
print({run: metrics[run]["chosen"] for run in cl.RUNS})

## 1. Inputs and choice of k

**Inputs** (`clustering.inputs`): own-cell fractions of water, trees, crops, built area and rangeland, plus mean elevation and slope, all standardised. Left out:
- `frac_snow` is always 0;
- `frac_flooded` and `frac_bare` are about 0;
- `slope_max` correlates with `slope_mean` at r = 0.95 (notebook 03).

PCA is reported but not used. 5 components reach 90 % of the variance, and the 7th explains only 0.03 %, because the fractions nearly sum to 1.

The scan below fits K-Means and GMM for k = 3 … 10 on both runs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for run, ls in (("validation", "--"), ("final", "-")):
    sc = pd.DataFrame(metrics[run]["scan"])
    for method, color in (("kmeans", "tab:blue"), ("gmm", "tab:orange")):
        s = sc[sc.method == method]
        axes[0].plot(s.k, s.silhouette, ls, marker="o", color=color, label=f"{method} {run}")
        axes[1].plot(s.k, s.davies_bouldin, ls, marker="o", color=color, label=f"{method} {run}")
    s = sc[sc.method == "kmeans"]
    axes[2].plot(s.k, s.inertia, ls, marker="o", color="tab:blue", label=f"kmeans {run}")
axes[0].set_title("Silhouette (higher is better)")
axes[1].set_title("Davies–Bouldin (lower is better)")
axes[2].set_title("K-Means inertia (elbow)")
for ax in axes:
    ax.axvline(7, color="grey", lw=0.8)
    ax.set_xlabel("k")
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "p5_k_scan.png", dpi=150)
pd.DataFrame(metrics["final"]["scan"]).pivot(
    index="k", columns="method", values=["silhouette", "davies_bouldin"]
).round(3)

**Choice: K-Means, k = 7.**
- **Silhouette:** for K-Means it plateaus at k = 5–8 (about 0.57–0.61) and drops at k = 9. k = 7 is the best or within 0.003 of the best in both runs: 0.585 for 2018 and 0.606 for 2023.
- **Two extra clusters:** compared with k = 5, k = 7 adds two clusters that are easy to interpret and appear in both years. One is steep rangeland (the Bannerghatta hills), the other the low western valley.
- **GMM:** much worse at every k (silhouette 0.26–0.36, Davies–Bouldin 1.6–2.9), and its BIC keeps falling up to k = 10 with no clear minimum. So K-Means is kept.

In [ ]:
def profile(run):
    r = rules[run]
    cols = [
        "frac_built",
        "frac_tree",
        "frac_crop",
        "frac_range",
        "frac_water",
        "elev_mean",
        "slope_mean",
    ]
    show = r[["share", *cols, "class_name", "rule", "flag"]].copy()
    show["share"] = (100 * show["share"]).round(1)
    return show.round(2)


fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, run in zip(axes, cl.RUNS, strict=True):
    r = rules[run]
    cols = [
        "frac_built",
        "frac_tree",
        "frac_crop",
        "frac_range",
        "frac_water",
        "elev_mean",
        "slope_mean",
    ]
    z = (r[cols] - r[cols].mean()) / r[cols].std()
    im = ax.imshow(z.to_numpy(), cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    for i in range(len(r)):
        for j, c in enumerate(cols):
            ax.text(
                j,
                i,
                f"{r[c].iloc[i]:.2f}" if c.startswith("frac") else f"{r[c].iloc[i]:.0f}",
                ha="center",
                va="center",
                fontsize=8,
            )
    ax.set_xticks(range(len(cols)), [c.replace("frac_", "") for c in cols], rotation=30)
    ax.set_yticks(
        range(len(r)),
        [
            f"{i}: {n} ({100 * s:.1f} %)"
            for i, n, s in zip(r.index, r["class_name"], r["share"], strict=True)
        ],
    )
    ax.set_title(f"Cluster means, {run} ({metrics[run]['year']}); colour = z-score")
fig.tight_layout()
fig.savefig(FIG / "p5_cluster_profiles.png", dpi=150)
profile("final")

## 2. What each cluster is

Ids are ordered by mean built share, so the same id can mean a different cluster in the other run (e.g. cropland is 4 in 2023 and 3 in 2018). The table below uses the **final run (2023)**. The 2018 clusters are the same seven types with similar shares.

| Cluster | Share | Profile | Interpretation | Class |
|---|---|---|---|---|
| 0 | 60.5 % | 96 % built, flat (2.6°) | The urban fabric: Electronic City, Bommasandra, Hosur Road, the southern city | **built-up** |
| 1 | 3.0 % | 50 % crop, 31 % built, 13 % rangeland; **lowest elevation (≈ 760 m)** | The low western valley: farmland with villages and new construction | **usable** (agricultural) |
| 2 | 3.1 % | 73 % rangeland, 13 % trees, **steep (13°)** | The Bannerghatta hills: dry deciduous forest and scrub that ESRI calls rangeland | **forest** |
| 3 | 2.4 % | 82 % water | Lakes and tanks (Bellandur, Madiwala, Hulimavu …) | **excluded** |
| 4 | 11.0 % | 88 % crop, flat | Cropland on the plateau | **usable** (agricultural) |
| 5 | 13.2 % | 88 % rangeland, flat (3.8°) | Open scrub, fallow and grassland on the urban fringe | **usable** |
| 6 | 6.9 % | 87 % trees | Tree cover: the Bannerghatta forest core, plantations, wooded campuses | **forest** |

Between 2018 and 2023, built-up grows from 52.9 % to 60.5 % of cells (cluster 0). Cropland shrinks from 16.5 % to 11.0 %.

## 3. Cluster → class rules and the exclusion mask

`src/classify/label.py` applies these rules in order to each cluster's mean fractions (thresholds in `config.yaml → labeling`):

1. **water + flooded ≥ 0.5** → excluded;
2. **built ≥ 0.3 and built is the largest land group** → built-up. A threshold alone would make the valley cluster built-up in 2023 (31 %) but not in 2018 (13 %);
3. **trees ≥ 0.5** → forest;
4. **trees + rangeland ≥ 0.5 on slopes ≥ 8°** → forest. This is the steep natural vegetation of the Bannerghatta hills;
5. otherwise **usable**, flagged *agricultural* when cropland dominates (decision D10).

Then, cell by cell, the **exclusion mask** (FR-5.5) sets class 255. A cell is masked if it is more than 50 % water / flooded / OSM lake, has slope > 15°, is more than 50 % nodata, or is more than 50 % inside a protected area (Bannerghatta NP). The validation run's mask uses only 2018/19 data. Saved as `outputs/exclusion_mask[_validation].tif`.

In [ ]:
for run in cl.RUNS:
    print(run, rules[run][["class_name", "rule", "flag"]].to_dict("index"))
excl = {run: lb.exclusion_table(cfg, run) for run in cl.RUNS}
pd.DataFrame(
    {
        run: e[["excl_wet", "excl_slope", "excl_nodata", "excl_protected", "excluded"]].sum()
        for run, e in excl.items()
    }
)

## 4. The 3-class maps

In [ ]:
def class_map(ax, run):
    g = gpd.GeoDataFrame(
        grid.merge(c7[run], on="cell_id"), geometry=gpd.points_from_xy(grid.x, grid.y), crs=cfg.crs
    )
    img = np.full((grid.row.max() + 1, grid.col.max() + 1), np.nan)
    img[g.row, g.col] = g.class_3.map({1: 0, 2: 1, 3: 2, 255: 3})
    ax.imshow(
        img,
        cmap=ListedColormap(list(CLASS_COLORS.values())),
        vmin=0,
        vmax=3,
        interpolation="nearest",
    )
    shares = c7[run].class_3.map(NAMES).value_counts(normalize=True)
    ax.set_title(
        f"{run} ({metrics[run]['year']}): "
        + ", ".join(f"{k} {100 * v:.0f} %" for k, v in shares.items()),
        fontsize=9,
    )
    ax.set_axis_off()


fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, run in zip(axes, cl.RUNS, strict=True):
    class_map(ax, run)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CLASS_COLORS.values()]
fig.legend(handles, [NAMES[k] for k in CLASS_COLORS], loc="lower center", ncol=4)
fig.tight_layout(rect=(0, 0.05, 1, 0.97))
fig.savefig(FIG / "p5_class_maps.png", dpi=150)

## 5. Agreement with land-cover references

Each 100 m cell's reference class is the collapsed class with the largest share of its 10 m pixels. **land** = cells that neither side calls excluded (3 classes); this is what the Phase 5 gate checks (≥ 80 %).

In [ ]:
ev = json.loads(eval_path(cfg).read_text())
rows = []
for run, res in ev.items():
    for ref, r in res.items():
        rows.append(
            {
                "run": run,
                "reference": ref,
                "agreement (all, 4 classes)": round(r["all"]["agreement"], 3),
                "agreement (land, 3 classes)": round(r["land"]["agreement"], 3),
            }
        )
display(pd.DataFrame(rows))
for ref, r in ev["final"].items():
    print(ref, "(final, land) rows = reference, columns = map")
    display(pd.DataFrame(r["land"]["confusion"]).T)

**Reading the numbers.**
- **ESRI:** agreement on land cells is 96.6 % in both runs. That's expected, because the clusters are built from ESRI fractions; the value of clustering lies in the sub-groups and the rules, not in reproducing ESRI.
- **WorldCover 2021:** agreement is only 64–71 %. The two products disagree with each other (P5.6): ESRI generalises urban areas (62 % built-up vs 33 % in WorldCover) and maps more scrub as rangeland, while WorldCover sees more trees (17 % vs 7 % forest).
- **Where the disagreement sits:** most of it is cells we call built-up that WorldCover calls usable or forest, i.e. tree-lined residential areas and campuses.
- **Limitation:** this is a known source limitation (Venter et al. 2022), recorded in `docs/data_sources.md` §7.

## 6. Visual check at 5 random locations

Five random grid cells (seed 5). Each panel shows a 1.2 km window with Esri World Imagery (current) on the left and the final 3-class map on the right. Needs internet for the basemap.

In [ ]:
import contextily as cx

pick = grid.merge(c7["final"], on="cell_id").sample(5, random_state=5)
half = 600
fig, axes = plt.subplots(5, 2, figsize=(10, 24))
g7 = gpd.GeoDataFrame(
    grid.merge(c7["final"], on="cell_id"),
    geometry=gpd.GeoSeries.from_xy(grid.x, grid.y, crs=cfg.crs).buffer(50, cap_style=3),
    crs=cfg.crs,
)
for (ax_img, ax_map), p in zip(axes, pick.itertuples(), strict=True):
    box = (p.x - half, p.x + half, p.y - half, p.y + half)
    for ax in (ax_img, ax_map):
        ax.set_xlim(box[0], box[1])
        ax.set_ylim(box[2], box[3])
        ax.set_xticks([])
        ax.set_yticks([])
    for _attempt in range(3):  # tile downloads can fail on a slow network
        try:
            cx.add_basemap(
                ax_img,
                crs=cfg.crs,
                source=cx.providers.Esri.WorldImagery,
                zoom=15,
                attribution_size=5,
            )
            break
        except Exception as e:  # offline
            err = e
    else:
        ax_img.text(
            0.5,
            0.5,
            f"basemap unavailable: {type(err).__name__}",
            transform=ax_img.transAxes,
            ha="center",
        )
    sub = g7.cx[box[0] : box[1], box[2] : box[3]]
    sub.plot(ax=ax_map, color=sub.class_3.map(CLASS_COLORS), edgecolor="none")
    for ax in (ax_img, ax_map):
        ax.plot(p.x, p.y, "k+", ms=14, mew=2)
    ax_img.set_title(f"cell {int(p.cell_id)}: satellite", fontsize=9)
    ax_map.set_title(
        f"cell {int(p.cell_id)}: {NAMES[int(p.class_3)]} (cluster {int(p.cluster_id)})", fontsize=9
    )
fig.tight_layout()
fig.savefig(FIG / "p5_visual_check.png", dpi=120)

**Verdict (checked 2026-10-05, Harsh): 4 of 5 clearly right, 1 debatable.**

| Cell | Satellite | Map | OK? |
|---|---|---|---|
| 26039 | Dense housing along a main road | built-up | ✅ |
| 44607 | Dense apartment blocks | built-up | ✅ |
| 34163 | Apartments with tree clumps and open plots | built-up (forest / usable patches nearby) | ✅ at 100 m |
| 52368 | Farm plots with scattered houses, a lake to the west | built-up (lake excluded) | ⚠️ ESRI generalises built-up over the village and farmland; the lake is right |
| 73799 | Rocky hill with dry scrub (Bannerghatta) | excluded (steep-vegetation cluster → forest, then the protected-area mask) | ✅ |

Cell 52368 shows the main weakness: ESRI's built-up class is broad on the urban fringe (the P5.6 finding). The fringe is where growth happens, so the report should mention it as a limitation.